<!-- CELL-00 -->
# DEBUG: FinBERT-FOMC + Beige Book Pipeline Inspection

Standalone diagnostic notebook. Loads 1 cached PDF, runs each pipeline
stage in isolation, exposes exactly where problems occur.

**Scop:** Fast iteration (<60 sec) without full backfill. Replaces the
scrape→fix→push→restart→backfill 10min loop.

**Stages inspected:**
1. PDF parsing (pdfplumber raw output)
2. PDF artifacts (hyphenation, page markers, headers)
3. Split into districts (split_pdf_into_sections)
4. Sentence segmentation (preprocess_beige_book)
5. Tokenization (FinBERT-FOMC BertTokenizer)
6. Token length distribution (sentences > 512 tokens)
7. Raw scoring on 5 sentences (probabilities, sums, Pydantic bug)
8. Full scoring on Boston district (label distribution)
9. Semantic spot-check: positive vs negative sentences vs model labels


In [ ]:
# CELL-01
# Idempotent environment bootstrap — self-sufficient for Colab + local

import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "macro_context_reader"
REPO_URL = "https://github.com/Inocenthhacker/macro_context_reader.git"

def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, check=True, capture=True):
    result = subprocess.run(cmd, capture_output=capture, text=True, shell=isinstance(cmd, str))
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\nSTDERR: {result.stderr[-500:]}")
    return result

# --- 1. Locate or clone repo ---
if _is_colab():
    repo_path = Path("/content") / REPO_NAME
    if not repo_path.exists():
        print(f"Cloning {REPO_URL} ...")
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            auth_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        except Exception:
            auth_url = REPO_URL
        _run(["git", "clone", auth_url, str(repo_path)])
    else:
        print(f"Repo exists at {repo_path}, pulling latest ...")
        _run(["git", "-C", str(repo_path), "pull", "--quiet"])
else:
    # Local: find repo root by walking up from notebook location
    nb_dir = Path.cwd()
    candidate = nb_dir
    while candidate != candidate.parent:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / REPO_NAME).exists():
            repo_path = candidate
            break
        candidate = candidate.parent
    else:
        raise RuntimeError(f"Cannot locate repo root from {nb_dir}")

print(f"Repo path: {repo_path}")

# --- 2. Install package editable ---
print("Installing package (editable) ...")
_run([sys.executable, "-m", "pip", "install", "-e", str(repo_path), "--quiet"])

# --- 3. Inject src into sys.path (Python 3.12 Colab .pth bug workaround) ---
src_path = str(repo_path / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Injected into sys.path: {src_path}")

# --- 4. Clear stale cached imports ---
stale = [m for m in sys.modules if m.startswith("macro_context_reader")]
for m in stale:
    del sys.modules[m]
if stale:
    print(f"Cleared {len(stale)} stale cached modules")

# --- 5. Load secrets (Colab Secrets with local .env fallback) ---
REQUIRED_SECRETS = ["FRED_API_KEY"]
OPTIONAL_SECRETS = ["HF_TOKEN", "GITHUB_TOKEN"]

def _load_secret(name: str) -> str | None:
    # Already in env?
    if os.environ.get(name):
        return os.environ[name]
    # Colab Secrets?
    if _is_colab():
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                os.environ[name] = val
                return val
        except Exception:
            pass
    # Local .env?
    env_file = repo_path / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith(f"{name}="):
                val = line.split("=", 1)[1].strip().strip("'\"")
                os.environ[name] = val
                return val
    return None

for secret in REQUIRED_SECRETS + OPTIONAL_SECRETS:
    val = _load_secret(secret)
    status = "OK" if val else ("MISSING" if secret in REQUIRED_SECRETS else "not set (optional)")
    print(f"  {secret}: {status}")

missing_required = [s for s in REQUIRED_SECRETS if not os.environ.get(s)]
if missing_required:
    print(f"\nWARNING: required secrets missing: {missing_required}")
    print("Add them via Colab Secrets (left sidebar key icon) or .env file locally.")

# --- 6. Verify import works ---
try:
    import macro_context_reader
    print(f"\nBootstrap complete. Package imported from: {macro_context_reader.__file__}")
except ImportError as e:
    raise RuntimeError(f"Bootstrap failed — package not importable after setup: {e}")


In [ ]:
# CELL-02
print("[CELL-02]")

from pathlib import Path
import requests

PDF_CACHE_DIR = Path("/content/macro_context_reader/data/economic_sentiment/raw/beige_book/pdf")
if not PDF_CACHE_DIR.exists():
    PDF_CACHE_DIR = Path("data/economic_sentiment/raw/beige_book/pdf")
PDF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_TEST_PDF = "BeigeBook_20240117.pdf"
CANONICAL_URL = f"https://www.federalreserve.gov/monetarypolicy/files/{CANONICAL_TEST_PDF}"

available_pdfs = sorted(PDF_CACHE_DIR.glob("*.pdf"))
if not available_pdfs:
    print(f"Cache empty. Downloading canonical test PDF: {CANONICAL_TEST_PDF}")
    canonical_path = PDF_CACHE_DIR / CANONICAL_TEST_PDF
    resp = requests.get(CANONICAL_URL, timeout=60)
    resp.raise_for_status()
    canonical_path.write_bytes(resp.content)
    print(f"  Downloaded {len(resp.content) // 1024} KB")
    available_pdfs = [canonical_path]

print(f"\nAvailable cached PDFs: {len(available_pdfs)}")
for p in available_pdfs[:5]:
    print(f"  {p.name} ({p.stat().st_size // 1024} KB)")

PDF_PATH = available_pdfs[-1]
print(f"\n-> Inspecting: {PDF_PATH.name}")


In [ ]:
# CELL-03
print("[CELL-03]")

import pdfplumber
import re

with pdfplumber.open(PDF_PATH) as pdf:
    n_pages = len(pdf.pages)
    full_text = "\n".join(p.extract_text() or "" for p in pdf.pages)

print(f"Pages: {n_pages}")
print(f"Total chars: {len(full_text)}")
print(f"Total non-empty lines: {sum(1 for l in full_text.split(chr(10)) if l.strip())}")
print(f"\n--- FIRST 1500 CHARS ---")
print(full_text[:1500])
print(f"\n--- LAST 800 CHARS ---")
print(full_text[-800:])


In [ ]:
# CELL-04
print("[CELL-04]")

# Hyphenation breaks (word- newline word)
hyphenated = re.findall(r"(\w+)-\n(\w+)", full_text)
print(f"Hyphenation breaks: {len(hyphenated)}")
print(f"  Examples: {hyphenated[:10]}")

# Page markers, headers, footers
page_refs = re.findall(r"Page \d+", full_text)
beige_headers = re.findall(r"Beige Book[^\n]{0,80}", full_text)
print(f"\nPage markers: {len(page_refs)} -- examples: {page_refs[:3]}")
print(f"Beige Book headers: {len(beige_headers)} -- examples: {beige_headers[:3]}")

# Unicode / encoding artifacts
weird_chars = set(c for c in full_text if ord(c) > 127 and c not in "\u2014\u2013\u2018\u201c\u201d\u2026")
print(f"\nNon-ASCII non-typographic chars: {weird_chars}")

# Federal Reserve Bank of X markers (district headers)
district_markers = re.findall(r"Federal Reserve Bank of [A-Z][a-z. ]+", full_text)
print(f"\nDistrict header markers found: {len(district_markers)}")
for d in district_markers:
    print(f"  - {d}")


In [ ]:
# CELL-05
print("[CELL-05]")

from macro_context_reader.economic_sentiment.scraper import (
    extract_text_from_pdf, split_pdf_into_sections,
)

extracted_text = extract_text_from_pdf(PDF_PATH)
sections = split_pdf_into_sections(extracted_text)
print(f"Total sections extracted: {len(sections)}")
print(f"\n{'Section':<25} {'Chars':>8} {'Status'}")
print("-" * 50)
EXPECTED_SECTIONS = [
    "national_summary", "Boston", "New York", "Philadelphia", "Cleveland",
    "Richmond", "Atlanta", "Chicago", "St. Louis", "Minneapolis",
    "Kansas City", "Dallas", "San Francisco",
]
for name in EXPECTED_SECTIONS:
    if name in sections:
        n_chars = len(sections[name])
        status = "OK" if n_chars > 1000 else "THIN" if n_chars > 0 else "EMPTY"
        print(f"{name:<25} {n_chars:>8} {status}")
    else:
        print(f"{name:<25} {'--':>8} MISSING")


In [ ]:
# CELL-06
print("[CELL-06]")

from macro_context_reader.economic_sentiment.preprocessor import preprocess_beige_book
from macro_context_reader.economic_sentiment.schemas import BeigeBookDocument
from datetime import datetime

# Construct minimal BeigeBookDocument for Boston
boston_doc = BeigeBookDocument(
    publication_date=datetime(2024, 1, 17),  # placeholder
    section_type="district_report",
    district="Boston",
    url="local://debug",
    raw_text=sections["Boston"],
)

boston_sentences = preprocess_beige_book(boston_doc)
print(f"Boston: {len(sections['Boston'])} chars -> {len(boston_sentences)} sentences")
print(f"Mean sentence length: {sum(len(s) for s in boston_sentences) / max(len(boston_sentences), 1):.0f} chars")
print(f"\nFirst 5 sentences:")
for i, s in enumerate(boston_sentences[:5]):
    print(f"  [{i}] [{len(s)} chars] {s[:200]}{'...' if len(s) > 200 else ''}")
print(f"\nLast 3 sentences:")
for i, s in enumerate(boston_sentences[-3:]):
    print(f"  [{i}] [{len(s)} chars] {s[:200]}{'...' if len(s) > 200 else ''}")


In [ ]:
# CELL-07
print("[CELL-07]")

from transformers import BertTokenizer
import numpy as np

tokenizer = BertTokenizer.from_pretrained("ZiweiChen/FinBERT-FOMC")
token_counts = [len(tokenizer.encode(s, add_special_tokens=True)) for s in boston_sentences]

counts = np.array(token_counts)
print(f"Boston sentences: {len(boston_sentences)}")
print(f"Token count stats:")
print(f"  min:  {counts.min()}")
print(f"  max:  {counts.max()}")
print(f"  mean: {counts.mean():.0f}")
print(f"  p50:  {np.percentile(counts, 50):.0f}")
print(f"  p90:  {np.percentile(counts, 90):.0f}")
print(f"  p99:  {np.percentile(counts, 99):.0f}")
print(f"\n>512 tokens (will be truncated): {(counts > 512).sum()} / {len(counts)} ({100*(counts > 512).mean():.1f}%)")
print(f">256 tokens: {(counts > 256).sum()} ({100*(counts > 256).mean():.1f}%)")

# Show longest 3 sentences
longest_idx = np.argsort(token_counts)[-3:][::-1]
print(f"\nLongest sentences:")
for idx in longest_idx:
    print(f"  [{token_counts[idx]} tokens] {boston_sentences[idx][:300]}...")


In [ ]:
# CELL-08
print("[CELL-08]")

from transformers import BertForSequenceClassification, pipeline
import torch

model = BertForSequenceClassification.from_pretrained("ZiweiChen/FinBERT-FOMC", num_labels=3)
device = 0 if torch.cuda.is_available() else -1
clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    truncation=True,
    max_length=512,
    device=device,
)

def _normalize_scores(raw):
    """Handle both old (list of dicts) and new (single dict) transformers API output."""
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        return [raw]
    raise TypeError(f"Unexpected result type: {type(raw)}")

import random
random.seed(42)
sample_indices = list(range(5)) + random.sample(range(5, len(boston_sentences)), min(5, len(boston_sentences)-5))
sample = [boston_sentences[i] for i in sample_indices]
results = clf(sample)

print(f"{'Idx':<4} {'Sum':<22} {'Top label':<10} {'Top score':<10} {'Sentence preview'}")
print("-" * 120)
max_sum = 0.0
for idx, sent, raw in zip(sample_indices, sample, results):
    scores = _normalize_scores(raw)
    total = sum(s["score"] for s in scores)
    top = max(scores, key=lambda s: s["score"])
    max_sum = max(max_sum, total)
    flag = " >1.0" if total > 1.0 else ""
    print(f"{idx:<4} {total:.10f}{flag:<8} {top['label']:<10} {top['score']:.4f}    {sent[:80]}")

print(f"\n-> Max sum probability: {max_sum:.10f}")
print(f"-> Max deviation from 1.0: {max_sum - 1.0:.2e}")
print(f"-> Pydantic bug present? {max_sum > 1.0}")
print(f"-> If False, CC-FIX-PYDANTIC in scorers/finbert_sentiment.py is effective")


In [ ]:
# CELL-09
print("[CELL-09]")

import time
t0 = time.time()
all_results = clf(boston_sentences, batch_size=32)
elapsed = time.time() - t0

label_counts = {"Positive": 0, "Negative": 0, "Neutral": 0}
for raw in all_results:
    scores = _normalize_scores(raw)
    top = max(scores, key=lambda s: s["score"])
    label_counts[top["label"]] = label_counts.get(top["label"], 0) + 1

total = sum(label_counts.values())
print(f"Boston scoring: {total} sentences in {elapsed:.1f}s ({total/elapsed:.0f} sent/sec)")
print(f"\nLabel distribution:")
for label, count in label_counts.items():
    pct = 100 * count / total if total else 0
    print(f"  {label:<10} {count:>4} ({pct:>5.1f}%)")

score = (label_counts["Positive"] - label_counts["Negative"]) / total if total else 0
print(f"\nBoston sentiment_score: {score:+.4f}")
print(f"  Range: [-1, +1] | -1 = all negative, +1 = all positive")
print(f"  Interpretation: {'POSITIVE economic activity' if score > 0.1 else 'NEGATIVE economic activity' if score < -0.1 else 'NEUTRAL/mixed'}")


In [ ]:
# CELL-10
print("[CELL-10]")

positive_keywords = ["increased", "grew", "expanded", "strong", "improved", "rose"]
negative_keywords = ["declined", "weakened", "contracted", "fell", "slowed", "dropped"]

print("=" * 80)
print("SEMANTIC SPOT-CHECK: Do FinBERT scores match human intuition?")
print("=" * 80)

for direction, keywords in [("POSITIVE expected", positive_keywords), ("NEGATIVE expected", negative_keywords)]:
    print(f"\n--- {direction} ---")
    matches = [(i, s) for i, s in enumerate(boston_sentences)
               if any(k in s.lower() for k in keywords)][:3]
    for idx, sent in matches:
        scores = _normalize_scores(all_results[idx])
        top = max(scores, key=lambda s: s["score"])
        score_dict = {s["label"]: s["score"] for s in scores}
        match_flag = "OK" if (("POSITIVE" in direction and top["label"] == "Positive") or
                              ("NEGATIVE" in direction and top["label"] == "Negative")) else "MISMATCH"
        print(f"  [{match_flag}] Top={top['label']} (P={score_dict.get('Positive', 0):.2f} / N={score_dict.get('Negative', 0):.2f} / Neu={score_dict.get('Neutral', 0):.2f})")
        print(f"       {sent[:200]}")
